# Chapter 26: Cost, banded and sparse systems

The code for this chapter from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/part-5/26-cost-sparse.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/26-cost-sparse.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

### Code for a figure

This draws a figure in the chapter. You don't need it for the rest of the notebook.

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

n = 12
A = np.eye(n) * 4
A[0, :] = A[:, 0] = 1
A[0, 0] = 4
B = A[::-1, ::-1]                       # number the unknowns in reverse

fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, M, title in [(axes[0], A, "A"), (axes[2], B, "A, reordered")]:
    ax.spy(M, markersize=4, color="#2153b8")
    ax.set_title(title, fontsize=10)
for ax, M in [(axes[1], A), (axes[3], B)]:
    P, L, U = scipy.linalg.lu(M)
    LU = np.abs(L) + np.abs(U) > 1e-12
    ax.spy(LU, markersize=4, color="#9c5208")
    ax.set_title(f"L and U: {LU.sum()} non-zeros", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## In code

SciPy's `scipy.sparse` module stores sparse matrices, and `scipy.sparse.linalg.spsolve` solves sparse systems. In MATLAB, `sparse` and `spdiags` create sparse matrices, and backslash automatically uses a sparse solver for them. The example solves $-u'' = 1$ on $0 < t < 1$, with $u = 0$ at both ends, by finite differences ([the chapter](https://trgoodale.github.io/matrix-book/part-6/33-finite-differences.html)): $n$ grid points give a tridiagonal system. Its exact solution is $u = t(1 - t)/2$.

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
import time

def thomas(a, d, c, b):
    """Solve a tridiagonal system: a below, d on and c above the diagonal."""
    n = len(d)
    cp, bp = np.zeros(n), np.zeros(n)
    cp[0], bp[0] = c[0] / d[0], b[0] / d[0]
    for i in range(1, n):
        m = d[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / m if i < n - 1 else 0
        bp[i] = (b[i] - a[i] * bp[i - 1]) / m
    x = np.zeros(n)
    x[-1] = bp[-1]
    for i in range(n - 2, -1, -1):
        x[i] = bp[i] - cp[i] * x[i + 1]
    return x

print(thomas([0, -1, -1, -1], [2, 2, 2, 2], [-1, -1, -1, 0], [1, 0, 0, 1]))

n = 100_000
h = 1 / (n + 1)
A = sp.diags([-1.0, 2.0, -1.0], [-1, 0, 1], shape=(n, n), format="csr")
b = h**2 * np.ones(n)
start = time.perf_counter()
u = spsolve(A, b)
print(f"sparse solve: {time.perf_counter() - start:.3f} s, {A.nnz} non-zeros")
t = np.arange(1, n + 1) * h
print(np.max(np.abs(u - t * (1 - t) / 2)))     # error

## Exercises

The code exercises from the [end of the chapter](https://trgoodale.github.io/matrix-book/part-5/26-cost-sparse.html#sec-cost-sparse-exercises). Worked solutions are in the notebook `solutions-part-5.ipynb`, and in the [book's solutions](https://trgoodale.github.io/matrix-book/appendices/solutions-5.html#sec-solutions-cost-sparse).

### Exercise 7

Time the solution of the tridiagonal system in the code above for $n = 500$, $1000$, $2000$ and $4000$, using (a) a dense matrix and `np.linalg.solve` or backslash on a full matrix, (b) the sparse solver. On a log–log plot, estimate the power of $n$ that each time grows like.

In [ ]:
# Your code here